# CSI Detection Overlay

**Source script:** `grdl/example/image_processing/sar/csi_detection_overlay.py`

This notebook demonstrates a complete SAR target detection workflow using sub-aperture dominance and CSI (Coherent Shape Index) composite visualization:

1. **Sub-aperture decomposition** — split full aperture into N independent looks
2. **Dominance feature** — identify scatterers concentrating energy in few looks
3. **Threshold detection** — statistical detector (mean + σ × threshold)
4. **CSI RGB composite** — encode scattering mechanism from sub-aperture phase
5. **Overlay detections** — visualize detection contours on CSI imagery

## Why This Matters

Sub-aperture **dominance** is a powerful detection feature:
- **High dominance** → target energy concentrates in 1-3 looks (man-made structures, vehicles)
- **Low dominance** → energy spreads evenly across all looks (natural clutter, vegetation)

The **CSI composite** provides physical insight:
- **Hue** → scattering mechanism (surface vs double-bounce vs volume)
- **Saturation** → polarimetric purity
- **Value** → backscatter intensity

## Data Setup

Set `SICD_PATH` in Cell 3 to point to any SICD NITF file on your system.

In [ ]:
"""GRDL Notebook Preamble — Environment validation and autoreload."""
import gc

try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except (NameError, AttributeError):
    pass

try:
    import grdl
except ImportError as exc:
    raise ImportError(
        "\n"
        "═══════════════════════════════════════════════════════════════\n"
        "  grdl is NOT installed in this Python environment.\n"
        "  Install with:  pip install -e '.[all]'\n"
        "  from the grdl repository root.\n"
        "═══════════════════════════════════════════════════════════════\n"
    ) from exc

print(f"grdl {grdl.__version__} — ready")

In [ ]:
from pathlib import Path
import numpy as np
from skimage.measure import label, regionprops
from skimage.morphology import opening, closing, footprint_rectangle

from grdl.IO import get_reader
from grdl.data_prep import ChipExtractor
from grdl.image_processing.sar import CSIProcessor, SublookDecomposition
from grdl.image_processing.sar.dominance import compute_dominance

# ════════════════════════════════════════════════════════════════════
# USER CONFIGURATION — Set your SICD file path here
# ════════════════════════════════════════════════════════════════════
SICD_PATH = Path("/path/to/your/sicd_file.nitf")  # ← CHANGE THIS to your SICD file
CHIP_SIZE = 5000          # pixels per side for center chip
NUM_LOOKS = 7             # number of azimuth sub-apertures
DIMENSION = 'azimuth'     # frequency axis to split
SIGMA = 3.0               # detection threshold in std devs above mean
SMOOTH_WIN = 7            # spatial smoothing kernel size
DOM_WINDOW = 3            # contiguous look window for dominance
MORPH_SIZE = 3            # morphological cleanup kernel size
CSI_NORM = 'log'          # CSI normalization: 'log', 'percentile', or 'none'

assert SICD_PATH.exists(), f"SICD file not found: {SICD_PATH}"
print(f"Target: {SICD_PATH.name}")

## 1. Read Center Chip

Extract a center chip from the SICD using `ChipExtractor` (index-only planning) and `get_reader` (factory pattern).

In [ ]:
with get_reader('sicd', SICD_PATH) as reader:
    meta = reader.metadata
    rows, cols = reader.get_shape()
    print(f"Image size: {rows} x {cols}")

    extractor = ChipExtractor(nrows=rows, ncols=cols)
    region = extractor.chip_at_point(
        rows // 2, cols // 2,
        row_width=CHIP_SIZE, col_width=CHIP_SIZE,
    )
    chip_h = region.row_end - region.row_start
    chip_w = region.col_end - region.col_start
    print(f"Center chip: [{region.row_start}:{region.row_end}, "
          f"{region.col_start}:{region.col_end}]")
    print(f"Chip dimensions: {chip_h} x {chip_w}")

    chip = reader.read_chip(
        region.row_start, region.row_end,
        region.col_start, region.col_end,
    )

print(f"\nChip shape: {chip.shape}, dtype: {chip.dtype}")

## 2. Sub-Aperture Decomposition

Split the full synthetic aperture into N non-overlapping azimuth sub-bands.
Each sub-look views the scene from a slightly different angle.

In [ ]:
sublook = SublookDecomposition(
    meta,
    num_looks=NUM_LOOKS,
    dimension=DIMENSION,
)

looks = sublook.decompose(chip)
print(f"Sublook stack: {looks.shape}, dtype: {looks.dtype}")

## 3. Dominance Feature Computation

The dominance feature measures how concentrated the energy is across sub-apertures:

$$
\text{dominance} = \max_{i} \left( \frac{\sum_{k=i}^{i+w-1} |z_k|^2}{\sum_{j=1}^{N} |z_j|^2} \right)
$$

where $w$ is the contiguous look window (`dom_window`).

High dominance → target (energy in 1-3 looks)  
Low dominance → clutter (energy spread evenly)

In [ ]:
print("Computing dominance...")
dominance = compute_dominance(
    looks,
    window_size=SMOOTH_WIN,
    dom_window=DOM_WINDOW,
)

print(f"Dominance shape: {dominance.shape}")
print(f"Dominance range: [{dominance.min():.4f}, {dominance.max():.4f}]")
print(f"Mean: {dominance.mean():.4f}, Std: {dominance.std():.4f}")

## 4. Statistical Thresholding

Detect pixels exceeding `mean + σ × SIGMA` as targets.
This is a constant false alarm rate (CFAR) detector variant.

In [ ]:
dom_mu = np.mean(dominance)
dom_std = np.std(dominance)
dom_thresh = dom_mu + SIGMA * dom_std

det_mask = dominance > dom_thresh

print(f"Threshold: {dom_thresh:.4f}")
print(f"  (μ={dom_mu:.4f}, σ={dom_std:.4f}, {SIGMA}σ above mean)")
print(f"Detection pixels: {det_mask.sum()} / {det_mask.size} "
      f"({100 * det_mask.sum() / det_mask.size:.2f}%)")

## 5. Morphological Cleanup

Remove speckle noise and fill small gaps using morphological opening and closing.

In [ ]:
fp = footprint_rectangle((MORPH_SIZE, MORPH_SIZE))
det_mask = opening(det_mask, footprint=fp)
det_mask = closing(det_mask, footprint=fp)

print(f"After morphological cleanup: {det_mask.sum()} pixels")

## 6. Label Connected Components

Group adjacent detection pixels into discrete objects.

In [ ]:
labeled = label(det_mask)
n_detections = labeled.max()

print(f"Detections: {n_detections} objects")

# Per-object summary
props = regionprops(labeled)
for p in props:
    print(f"  Object {p.label}: centroid=({p.centroid[0]:.0f}, {p.centroid[1]:.0f}), "
          f"area={p.area} px")

## 7. CSI (Coherent Shape Index) Composite

The CSI processor converts sub-aperture phase into an HSV color encoding:
- **Hue** → scattering mechanism phase signature
- **Saturation** → spectral purity
- **Value** → backscatter intensity

Output is an RGB image ready for display.

In [ ]:
import gc

print("Computing CSI composite...")
csi_proc = CSIProcessor(
    meta,
    dimension=DIMENSION,
    normalization=CSI_NORM,
)
csi_rgb = csi_proc.apply(chip)

print(f"CSI RGB shape: {csi_rgb.shape}, dtype: {csi_rgb.dtype}")
print(f"CSI range: [{csi_rgb.min()}, {csi_rgb.max()}]")

# Free chip and looks (no longer needed for visualization)
del chip, looks
gc.collect()

## 8. Visualization with napari

Display the CSI composite with detection contours overlaid as vector shapes.

**napari layers**:
- Base: CSI RGB composite
- Overlay 1: Detection mask (binary)
- Overlay 2: Labeled regions (colored by object ID)
- Overlay 3: Dominance feature map (raw values)

Toggle layer visibility to compare detections against CSI and dominance.

In [ ]:
import napari
from skimage.measure import find_contours

viewer = napari.Viewer(title="CSI Detection Overlay")

# CSI RGB composite as base layer
viewer.add_image(
    csi_rgb,
    name="CSI Composite",
    rgb=True,
)

# Detection mask
viewer.add_image(
    det_mask.astype(np.uint8),
    name="Detection Mask",
    colormap="green",
    opacity=0.3,
    visible=False,
)

# Labeled regions (each object gets a different color)
viewer.add_labels(
    labeled,
    name=f"Detections ({n_detections} objects)",
    opacity=0.5,
)

# Dominance feature map
viewer.add_image(
    dominance,
    name="Dominance Feature",
    colormap="hot",
    contrast_limits=[dom_mu, dom_thresh],
    visible=False,
)

# Simplify viewer UI - hide unnecessary controls
viewer.window._qt_viewer.dockLayerControls.setVisible(False)  # Hide layer controls dock
viewer.window._qt_viewer.dockConsole.setVisible(False)  # Hide console
try:
    viewer.window._qt_viewer.activityDock.setVisible(False)  # Hide activity dock if present
except AttributeError:
    pass

print(f"✓ Napari viewer opened with {len(viewer.layers)} layers")
print(f"Detected {n_detections} objects")
print("Toggle layer visibility to explore CSI, dominance, and detections.")
print("\nExecution paused — close the napari window to continue and free memory...")
napari.run()

print("\n✓ Viewer closed — cleaning up memory...")
del csi_rgb, dominance, det_mask, labeled, dom_mu, dom_std, dom_thresh, props, viewer
gc.collect()
print("✓ Memory cleanup complete")

---

## Summary

This notebook replaces `grdl/example/image_processing/sar/csi_detection_overlay.py`.

Key patterns:
- `get_reader('sicd', path)` — factory pattern for format-agnostic IO
- `SublookDecomposition` → azimuth sub-aperture splitting
- `compute_dominance` → energy concentration feature
- `CSIProcessor` → HSV color encoding of scattering mechanism
- `skimage.morphology` → speckle noise cleanup
- `skimage.measure.label` → connected component analysis
- napari labels layer → interactive detection visualization

**Physical interpretation**:
- High dominance + bright in CSI → man-made structure (building, vehicle, infrastructure)
- Low dominance + dim in CSI → natural clutter (vegetation, rough terrain)
- CSI color → scattering mechanism (red=double-bounce, green=volume, blue=surface)